In [3]:
import sys
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# ============================================================
# 0. THIẾT LẬP ĐƯỜNG DẪN
# ============================================================

# Thêm thư mục gốc của dự án vào sys.path
ROOT = Path.cwd()

if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.baseline import (
    BASELINE_COLUMNS,
    get_common_eval_timestamps,
    predict_baselines,
)

from src.data import (
    load_hourly_data,
    load_splits,
    fit_and_save_scaler,
)

from src.metrics import (
    calculate_d_train,
    evaluate_forecast,
    evaluate_by_group,
)

# ============================================================
# 1. THIẾT LẬP CHUNG
# ============================================================

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

# Chuỗi dữ liệu MT_168
CUSTOMER_COL = "MT_168"

# Thư mục lưu kết quả riêng theo chuỗi dữ liệu
RESULTS_DIR = Path("results") / CUSTOMER_COL

TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

# Scaler lưu trong folder models chung
MODELS_DIR = Path("models")

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================
# 2. ĐỌC DỮ LIỆU VÀ CÁC TẬP ĐÃ CHIA
# ============================================================

full_df = load_hourly_data(customers=[CUSTOMER_COL])
splits = load_splits(customers=[CUSTOMER_COL])

full_series = full_df[CUSTOMER_COL].astype(float)

train_series = splits["train"][CUSTOMER_COL].astype(float)
val_series = splits["validation"][CUSTOMER_COL].astype(float)
test_series = splits["test"][CUSTOMER_COL].astype(float)

print(f"Chuỗi dữ liệu: {CUSTOMER_COL}")

print(
    f"Train: {len(train_series)} giờ | "
    f"Validation: {len(val_series)} giờ | "
    f"Test: {len(test_series)} giờ"
)

# Kiểm tra timestamp
for split_name, split_series in [
    ("Train", train_series),
    ("Validation", val_series),
    ("Test", test_series),
]:
    if not split_series.index.isin(full_series.index).all():
        raise ValueError(
            f"{split_name} có timestamp không tồn tại trong full_series."
        )

# ============================================================
# 3. TÍNH d_train TRÊN TRAIN GỐC
# ============================================================

d_train = calculate_d_train(train_series.to_numpy())

if not np.isfinite(d_train) or d_train <= 0:
    raise ValueError(
        f"d_train không hợp lệ ({d_train}); không thể tính MASE."
    )

print(f"d_train = {d_train:.6f}")

# ============================================================
# 4. FIT SCALER TRÊN TRAIN
# ============================================================
# Baseline vẫn dự báo trên giá trị điện năng gốc.
# Scaler chỉ được fit trên tập Train.

scaler = fit_and_save_scaler(
    train_df=splits["train"],
    customer=CUSTOMER_COL,
    models_dir=str(MODELS_DIR),
)

print(f"Đã lưu scaler riêng cho {CUSTOMER_COL}")

# ============================================================
# 5. TẠO TIMESTAMP ĐÁNH GIÁ
# ============================================================

print("Đang tạo timestamp đánh giá...", flush=True)

val_eval_timestamps = get_common_eval_timestamps(
    full_series,
    val_series.index,
)

test_eval_timestamps = get_common_eval_timestamps(
    full_series,
    test_series.index,
)

print("Số timestamp Validation:", len(val_eval_timestamps))
print("Số timestamp Test:", len(test_eval_timestamps))

if len(val_eval_timestamps) == 0:
    raise ValueError("Không có timestamp hợp lệ trong Validation.")

if len(test_eval_timestamps) == 0:
    raise ValueError("Không có timestamp hợp lệ trong Test.")

# ============================================================
# 6. DỰ BÁO VÀ ĐÁNH GIÁ VALIDATION
# ============================================================

print("\nĐang dự báo Validation...", flush=True)

val_predictions = predict_baselines(
    full_series,
    val_eval_timestamps,
)

print("Đã dự báo xong Validation.", flush=True)
print("Số dòng:", len(val_predictions))
print("Các cột:", val_predictions.columns.tolist())

display(val_predictions.head())

# Kiểm tra các cột baseline
available_models = [
    col for col in BASELINE_COLUMNS
    if col in val_predictions.columns
]

if not available_models:
    raise ValueError("Không tìm thấy cột dự báo baseline.")

# Đánh giá Validation
val_metrics = {}

for col in available_models:
    y_true = val_predictions["y_true"].to_numpy(dtype=float)
    y_pred = val_predictions[col].to_numpy(dtype=float)

    val_metrics[col] = evaluate_forecast(
        y_true,
        y_pred,
        d_train=d_train,
    )

df_val_results = pd.DataFrame(val_metrics).T
df_val_results.index.name = "model"

df_val_results = df_val_results.sort_values("MAE")

print("\n=== KẾT QUẢ VALIDATION ===")
display(df_val_results)

# Chọn baseline tốt nhất theo Validation MAE
best_model = df_val_results["MAE"].idxmin()

print("\nBaseline tốt nhất:", best_model)

# ============================================================
# 7. DỰ BÁO VÀ ĐÁNH GIÁ TEST
# ============================================================

print("\nĐang dự báo Test...", flush=True)

test_predictions = predict_baselines(
    full_series,
    test_eval_timestamps,
)

print("Đã dự báo xong Test.", flush=True)
print("Số dòng:", len(test_predictions))

display(test_predictions.head())

# Đánh giá Test
test_metrics_all = {}

for col in available_models:
    y_true_test = test_predictions["y_true"].to_numpy(dtype=float)
    y_pred_test = test_predictions[col].to_numpy(dtype=float)

    test_metrics_all[col] = evaluate_forecast(
        y_true_test,
        y_pred_test,
        d_train=d_train,
    )

df_test_results = pd.DataFrame(test_metrics_all).T
df_test_results.index.name = "model"

print("\n=== KẾT QUẢ TEST ===")
display(df_test_results)

print("\n=== BASELINE ĐƯỢC CHỌN TRÊN VALIDATION ===")
display(df_test_results.loc[[best_model]])

# ============================================================
# 8. PHÂN TÍCH SAI SỐ THEO GIỜ VÀ THỨ
# ============================================================

y_true_test = test_predictions[
    "y_true"
].to_numpy(dtype=float)

y_pred_best = test_predictions[
    best_model
].to_numpy(dtype=float)

test_ts = pd.DatetimeIndex(
    pd.to_datetime(test_predictions["timestamp"])
)

# Chuẩn bị dữ liệu theo API evaluate_by_group
df_group_eval = test_predictions.copy()

df_group_eval["timestamp"] = pd.to_datetime(
    df_group_eval["timestamp"]
)

df_error_by_hour = evaluate_by_group(
    df_group_eval,
    pred_col=best_model,
    group_by="hour",
)

df_error_by_dow = evaluate_by_group(
    df_group_eval,
    pred_col=best_model,
    group_by="dayofweek",
)

print("\n=== SAI SỐ THEO GIỜ ===")
print(df_error_by_hour)

print("\n=== SAI SỐ THEO THỨ TRONG TUẦN ===")
print(df_error_by_dow)

# ============================================================
# 9. TOP 10 DỰ BÁO CÓ SAI SỐ LỚN NHẤT
# ============================================================

test_predictions_copy = test_predictions.copy()

test_predictions_copy["abs_error"] = np.abs(
    y_true_test - y_pred_best
)

top10_worst = test_predictions_copy.sort_values(
    "abs_error",
    ascending=False,
).head(10)

top10_path = (
    TABLES_DIR /
    f"top10_worst_errors_{CUSTOMER_COL}.csv"
)

top10_worst.to_csv(
    top10_path,
    index=False,
)

# ============================================================
# 10. BIỂU ĐỒ SO SÁNH BASELINE TRÊN VALIDATION
# ============================================================

ax = df_val_results[
    ["MAE", "RMSE"]
].plot(
    kind="bar",
    figsize=(9, 5),
)

ax.set_title(
    f"So sánh MAE và RMSE của các Baseline - {CUSTOMER_COL} - Validation"
)

ax.set_xlabel("Mô hình Baseline")
ax.set_ylabel("Sai số")

plt.xticks(rotation=0)
plt.tight_layout()

val_comparison_path = (
    FIGURES_DIR /
    f"baseline_val_comparison_{CUSTOMER_COL}.png"
)

plt.savefig(
    val_comparison_path,
    dpi=300,
)

plt.close()

# ============================================================
# 11. BIỂU ĐỒ THỰC TẾ VÀ DỰ BÁO TRÊN TEST
# ============================================================

sample_hours = min(
    168,
    len(test_predictions),
)

plt.figure(figsize=(14, 5))

plt.plot(
    test_ts[:sample_hours],
    y_true_test[:sample_hours],
    label="Thực tế",
    color="black",
    linewidth=1.5,
)

plt.plot(
    test_ts[:sample_hours],
    y_pred_best[:sample_hours],
    label=f"Dự báo: {best_model}",
    color="red",
    linestyle="--",
    linewidth=1.2,
)

plt.title(
    f"Dự báo một tuần đầu của Test - {CUSTOMER_COL} - {best_model}"
)

plt.xlabel("Thời gian")
plt.ylabel("Phụ tải (kW)")

plt.legend()
plt.tight_layout()

test_timeseries_path = (
    FIGURES_DIR /
    f"baseline_test_timeseries_{CUSTOMER_COL}.png"
)

plt.savefig(
    test_timeseries_path,
    dpi=300,
)

plt.close()

# ============================================================
# 12. BIỂU ĐỒ MAE THEO GIỜ TRONG NGÀY
# ============================================================

plt.figure(figsize=(9, 4))

sns.barplot(
    data=df_error_by_hour,
    x="hour",
    y="MAE",
    color="skyblue",
)

plt.title(
    f"MAE theo giờ trong ngày - {CUSTOMER_COL} - {best_model}"
)

plt.xlabel("Giờ trong ngày (0–23)")
plt.ylabel("MAE")

plt.xticks(range(0, 24))
plt.tight_layout()

error_by_hour_path = (
    FIGURES_DIR /
    f"baseline_error_by_hour_{CUSTOMER_COL}.png"
)

plt.savefig(
    error_by_hour_path,
    dpi=300,
)

plt.close()

# ============================================================
# 13. XUẤT CÁC FILE KẾT QUẢ
# ============================================================

# Kết quả Validation
val_results_path = (
    TABLES_DIR /
    f"baseline_results_{CUSTOMER_COL}.csv"
)

df_val_results.to_csv(
    val_results_path,
    index=True,
)

# Kết quả Test
test_results_path = (
    TABLES_DIR /
    f"baseline_test_results_{CUSTOMER_COL}.csv"
)

df_test_results.to_csv(
    test_results_path,
    index=True,
)

# Dự báo Validation
val_predictions_path = (
    RESULTS_DIR /
    f"baseline_predictions_validation_{CUSTOMER_COL}.csv"
)

val_predictions.to_csv(
    val_predictions_path,
    index=False,
)

# Dự báo Test
test_predictions_path = (
    RESULTS_DIR /
    f"baseline_predictions_test_{CUSTOMER_COL}.csv"
)

test_predictions.to_csv(
    test_predictions_path,
    index=False,
)

# File dự báo gộp Validation + Test
val_output = val_predictions.copy()
val_output.insert(0, "split", "validation")

test_output = test_predictions.copy()
test_output.insert(0, "split", "test")

all_predictions = pd.concat(
    [val_output, test_output],
    ignore_index=True,
)

all_predictions_path = (
    RESULTS_DIR /
    f"baseline_predictions_{CUSTOMER_COL}.csv"
)

all_predictions.to_csv(
    all_predictions_path,
    index=False,
)

# Danh sách timestamp đánh giá chung
eval_timestamps_df = pd.concat(
    [
        pd.DataFrame({
            "split": "validation",
            "timestamp": val_eval_timestamps,
        }),
        pd.DataFrame({
            "split": "test",
            "timestamp": test_eval_timestamps,
        }),
    ],
    ignore_index=True,
)

eval_timestamps_path = (
    RESULTS_DIR /
    f"eval_timestamps_{CUSTOMER_COL}.csv"
)

eval_timestamps_df.to_csv(
    eval_timestamps_path,
    index=False,
)

# Sai số theo giờ
error_by_hour_path_csv = (
    TABLES_DIR /
    f"error_by_hour_{CUSTOMER_COL}.csv"
)

df_error_by_hour.to_csv(
    error_by_hour_path_csv,
    index=False,
)

# Sai số theo thứ trong tuần
error_by_dow_path_csv = (
    TABLES_DIR /
    f"error_by_dayofweek_{CUSTOMER_COL}.csv"
)

df_error_by_dow.to_csv(
    error_by_dow_path_csv,
    index=False,
)

# ============================================================
# 14. THÔNG BÁO HOÀN THÀNH
# ============================================================

print("\n==============================================")
print("HOÀN THÀNH THÍ NGHIỆM BASELINE")
print("Chuỗi dữ liệu:", CUSTOMER_COL)
print("Validation:", val_results_path)
print("Test:", test_results_path)
print("Dự báo gộp:", all_predictions_path)
print("Timestamp:", eval_timestamps_path)
print("Top 10 sai số:", top10_path)
print("Biểu đồ:", FIGURES_DIR)
print("==============================================")

Chuỗi dữ liệu: MT_168
Train: 27720 giờ | Validation: 3672 giờ | Test: 3672 giờ
d_train = 14.222778
Đã lưu scaler: models\scaler_MT_168.pkl
Đã lưu scaler riêng cho MT_168
Đang tạo timestamp đánh giá...
Số timestamp Validation: 3672
Số timestamp Test: 3672

Đang dự báo Validation...
Đã dự báo xong Validation.
Số dòng: 3672
Các cột: ['timestamp', 'y_true', 'y_last_value', 'y_ma3', 'y_snaive24', 'y_snaive168']


,timestamp,y_true,y_last_value,y_ma3,y_snaive24,y_snaive168
0,2014-03-01 01:00:00,83.227991,108.425508,115.015989,73.905192,68.261851
1,2014-03-01 02:00:00,58.941874,83.227991,102.424755,77.011851,66.566027
2,2014-03-01 03:00:00,59.506208,58.941874,83.531791,77.296840,64.308691
3,2014-03-01 04:00:00,54.424379,59.506208,67.225357,69.672686,56.963883
4,2014-03-01 05:00:00,56.399549,54.424379,57.624153,67.979684,59.221219



=== KẾT QUẢ VALIDATION ===


,MAE,RMSE,MASE,N_samples
model,,,,
y_snaive24,6.918478,9.518322,0.486437,3672.0
y_snaive168,8.438382,11.154073,0.593301,3672.0
y_last_value,12.116310,19.193662,0.851895,3672.0
y_ma3,18.955760,27.260818,1.332775,3672.0



Baseline tốt nhất: y_snaive24

Đang dự báo Test...
Đã dự báo xong Test.
Số dòng: 3672


,timestamp,y_true,y_last_value,y_ma3,y_snaive24,y_snaive168
0,2014-08-01 01:00:00,121.001693,135.211625,134.458239,97.415350,77.296840
1,2014-08-01 02:00:00,76.732506,121.001693,130.475922,74.190181,68.544018
2,2014-08-01 03:00:00,71.365688,76.732506,110.981941,71.932844,69.390519
3,2014-08-01 04:00:00,69.957675,71.365688,89.699962,72.215011,66.566027
4,2014-08-01 05:00:00,70.234199,69.957675,72.685290,71.365688,66.001693



=== KẾT QUẢ TEST ===


,MAE,RMSE,MASE,N_samples
model,,,,
y_last_value,12.211446,18.945211,0.858584,3672.0
y_ma3,18.900182,27.695984,1.328867,3672.0
y_snaive24,7.861975,11.808074,0.552773,3672.0
y_snaive168,9.576674,13.198552,0.673334,3672.0



=== BASELINE ĐƯỢC CHỌN TRÊN VALIDATION ===


,MAE,RMSE,MASE,N_samples
model,,,,
y_snaive24,7.861975,11.808074,0.552773,3672.0



=== SAI SỐ THEO GIỜ ===
    hour        MAE       RMSE  N_samples
0      0   8.356054  12.137537        153
1      1  12.600879  15.598140        153
2      2   8.217571  12.522203        153
3      3   4.681594   8.221736        153
4      4   3.879760   6.306667        153
5      5   3.597132   5.310741        153
6      6   5.500450   8.906703        153
7      7  12.718836  16.850007        153
8      8   9.114475  13.351534        153
9      9   9.060015  13.152399        153
10    10   6.762013  10.668704        153
11    11   8.277213  12.303441        153
12    12   9.570239  14.220991        153
13    13  10.163417  13.957561        153
14    14   9.474100  12.932627        153
15    15   9.508402  13.441776        153
16    16   7.838803  11.532086        153
17    17   8.937964  11.983433        153
18    18   7.581441  11.730171        153
19    19   8.425453  12.055391        153
20    20   6.849024  10.752671        153
21    21   5.268096   8.599692        153
22    22 